# 00_config
Shared settings + one-time table setup for both pipelines. Safe to `%run` many times: tables are created only if missing, and each source's control row is seeded only if it is missing.

Both pipelines use the **same stages and the same control table**: *copy zips -> unzip -> load*. They differ in frequency and in how the unzipped JSON is loaded.

| | **NPHC** | **HSAR** |
|---|---|---|
| Frequency | monthly | daily |
| Zip name | `<table>_<start>_<MID>_<end>_<hhmmss>.zip` | `<name>_<yyyymmdd>_<hhmmss>.zip` |
| Folder date (`raw/<date>/`, `processed/<date>/`) | MID date | date in the name |
| Load into target | one metadata row per JSON file (+ `zip_file`) | Auto Loader: one row per JSON file with the full JSON content (+ `zip_file`) |
| Entry notebook | `03_main_nphc` | `03_main_hsar` |

In [0]:
import os
import re
import uuid
import zipfile
import traceback
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, LongType, TimestampType
)

### Settings - shared

In [0]:
catalog = "iics"
schema  = "casemix"

control_table  = f"{catalog}.{schema}.zip_files_and_json_control_test"   # one row per source (NPHC, HSAR)
log_table      = f"{catalog}.{schema}.zip_files_execution_log_test"      # COPY / UNZIP log (both) + HSAR_LOAD log
load_log_table = f"{catalog}.{schema}.json_large_files_load_log"         # NPHC per-JSON-file ingestion log

EXECUTED_BY = spark.sql("SELECT current_user()").first()[0]

# TEST ONLY: True drops the control table, both log tables and ALL target tables (and HSAR checkpoints), then starts clean.
RESET_TABLES = False

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {control_table}")
spark.sql(f"DROP TABLE IF EXISTS {log_table}")
spark.sql(f"DROP TABLE IF EXISTS {load_log_table}")

### Settings - NPHC (monthly)

In [0]:
# <table_name>_<start yyyymmdd>_<MID yyyymmdd>_<end yyyymmdd>_<hhmmss>.zip  (or .json)
# e.g. case_data_001_20220605_20220620_20260101_235954.zip  ->  mid_date = 20220620, end month = 202601
NPHC_FILE_NAME_PATTERN = re.compile(
    r"^(?P<table_name>.+?)_(?P<start_date>\d{8})_(?P<mid_date>\d{8})_(?P<end_date>\d{8})_\d{6}\.(zip|json)$",
    re.IGNORECASE,
)

### Settings - HSAR (daily)

In [0]:
# names end with _<yyyymmdd>_<hhmmss>.<ext>   (zip and json alike)
# the date is the folder name under raw/ and processed/, and gives dt_data_file and the year/month partitions
HSAR_FILE_DATE_PATTERN = r"_(\d{8})_\d{6}\.[A-Za-z]+$"

HSAR_MAX_FILES_PER_TRIGGER = 500     # Auto Loader batch size

### Reset (test only)

In [0]:
if RESET_TABLES:
    if spark.catalog.tableExists(control_table):
        for r in spark.sql(f"SELECT src_system, unzip_tgt_path, tgt_catalog, tgt_schema, tgt_table FROM {control_table}").collect():
            spark.sql(f"DROP TABLE IF EXISTS {r.tgt_catalog}.{r.tgt_schema}.{r.tgt_table}")
            if r.src_system == "HSAR":                       # without this Auto Loader would think the files are already loaded
                dbutils.fs.rm(r.unzip_tgt_path.rstrip("/") + "/checkpoint/", True)
    for t in (control_table, log_table, load_log_table):
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    print("Tables dropped")

### Control table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {control_table} (
    s_no                        INT NOT NULL,
    src_system                  STRING NOT NULL,   -- NPHC | HSAR : decides which pipeline reads the row
    zip_source                  STRING NOT NULL,
    zip_source_path             STRING NOT NULL,   -- folder with the .zip files
    zip_tgt                     STRING NOT NULL,
    zip_tgt_path                STRING NOT NULL,   -- raw/        : copied zips, in <date>/ folders
    unzip_tgt_path              STRING NOT NULL,   -- processed/  : unzipped files, in <date>/<zip_name>/ folders
    file_type                   STRING NOT NULL,
    tgt_catalog                 STRING NOT NULL,
    tgt_schema                  STRING NOT NULL,
    tgt_table                   STRING NOT NULL,
    versioning_partition_filed  STRING NOT NULL,
    is_full_load                INT NOT NULL,      -- 1 = load everything again
    is_incremental_load         INT NOT NULL,      -- 1 = load only what is new / missing
    is_active                   INT NOT NULL
)
USING DELTA
""")

### Log tables
`log_table`: **COPY / UNZIP** records for both pipelines, plus **HSAR_LOAD** (one per dated folder). `load_log_table`: NPHC JSON ingestion, one row per JSON file including its `zip_file`.

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id",      StringType(),    True),
    StructField("stage",       StringType(),    True),   # COPY | UNZIP | HSAR_LOAD
    StructField("s_no",        IntegerType(),   True),
    StructField("source",      StringType(),    True),   # src_system
    StructField("file_name",   StringType(),    True),   # HSAR_LOAD: the dated folder
    StructField("source_path", StringType(),    True),
    StructField("target_path", StringType(),    True),   # HSAR_LOAD: the target table
    StructField("status",      StringType(),    True),   # SUCCESS | FAILED | SKIPPED
    StructField("message",     StringType(),    True),
    StructField("start_time",  TimestampType(), True),
    StructField("end_time",    TimestampType(), True),
])

LOAD_LOG_SCHEMA = StructType([
    StructField("run_id",           StringType(),    True),
    StructField("src",              StringType(),    True),
    StructField("src_path",         StringType(),    True),
    StructField("zip_file",         StringType(),    True),
    StructField("file_name",        StringType(),    True),
    StructField("file_path",        StringType(),    True),
    StructField("file_size_mb",     DoubleType(),    True),
    StructField("file_modified_ms", LongType(),      True),
    StructField("target_table",     StringType(),    True),
    StructField("status",           StringType(),    True),   # SUCCESS | FAILED
    StructField("error_type",       StringType(),    True),
    StructField("error_message",    StringType(),    True),
    StructField("started_at",       TimestampType(), True),
    StructField("ended_at",         TimestampType(), True),
    StructField("executed_by",      StringType(),    True),
])

def create_log_table(name, struct):
    cols = ",\n    ".join(f"{f.name} {f.dataType.simpleString().upper()}" for f in struct.fields)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {name} (\n    {cols}\n) USING DELTA")

create_log_table(log_table, LOG_SCHEMA)
create_log_table(load_log_table, LOAD_LOG_SCHEMA)

### Control rows
Both rows are **incremental** (`is_full_load = 0`, `is_incremental_load = 1`). The first run finds nothing copied yet, so it loads everything; later runs load only what is new or missing.

In [0]:
%sql
INSERT INTO iics.casemix.zip_files_and_json_control_test (
    s_no,
    src_system,
    zip_source,
    zip_source_path,
    zip_tgt,
    zip_tgt_path,
    unzip_tgt_path,
    file_type,
    tgt_catalog,
    tgt_schema,
    tgt_table,
    versioning_partition_filed,
    is_full_load,
    is_incremental_load,
    is_active
)
VALUES
(
    1, 'NPHC', 's3',
    's3://cittabase-s3-test/Databricks-poc/zip_files/',
    'databricks_volume',
    '/Volumes/iics/casemix/history/casemix/zip_files/raw/',
    '/Volumes/iics/casemix/history/casemix/zip_files/processed/',
    'json', 'iics', 'casemix', 'nphc_json_test',
    'file_name', 0, 1, 1
),
(
    2, 'HSAR', 's3',
    's3://cittabase-s3-test/Databricks-poc/hsar_zip_files/',
    'databricks_volume',
    '/Volumes/iics/casemix/history/casemix_hsar/zip_files/raw/',
    '/Volumes/iics/casemix/history/casemix_hsar/zip_files/processed/',
    'json', 'iics', 'casemix', 'hsar_json_test',
    'n/a', 1, 0, 1
);

In [0]:
%sql
drop table if exists iics.casemix.nphc_json_test

In [0]:
%sql
drop table if exists iics.casemix.hsar_json_test